In [96]:
import os
import pandas as pd
import numpy as np

REPO_DIR = os.path.join("/Users/haya1/Documents/LanguageModel_Labels/congressional_bills/")
# REPO_DIR = "."
os.chdir(REPO_DIR)
data_dir = os.path.join(REPO_DIR, "Data/Prediction_run1")
temp_dir = os.path.join(REPO_DIR, "Temp/Prediction_run1")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def euclidean_distance(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.linalg.norm(a - b)

bills_llm_completion = pd.read_csv(os.path.join(data_dir, f"bills_llm_completion.csv"))

print(len(bills_llm_completion))


39999


In [97]:
bills_llm_completion["TextSimilarity"] = bills_llm_completion["DescriptionClean"] == bills_llm_completion["DescriptionLLMClean"]

In [98]:
def decode_embed_responses(responses):
    responses_decoded = []
    for _, response in responses.iterrows(): 
        response_out =  response.response['body']['data'][0]
        responses_decoded.append({
            "custom_id": response["custom_id"],
            "Embeddings": response_out['embedding'],
            "Tokens": int(response.response["body"]["usage"]["prompt_tokens"])
        })
    return(pd.json_normalize(responses_decoded))
description_embeddings = decode_embed_responses(pd.read_json(os.path.join(temp_dir, 'Embeddings/Responses/responses_DescriptionClean.jsonl'), lines=True))
description_llm_embeddings = decode_embed_responses(pd.read_json(os.path.join(temp_dir, 'Embeddings/Responses/responses_DescriptionLLMClean.jsonl'), lines=True))

description_embeddings.rename(columns={
    'custom_id': 'ID',
    'Embeddings': 'DescriptionCleanEmbed',
    'Tokens': 'DescriptionCleanTokens'
    }, inplace=True)

description_llm_embeddings.rename(columns={
    'custom_id': 'ID',
    'Embeddings': 'DescriptionLLMCleanEmbed',
    'Tokens': 'DescriptionLLMCleanTokens'
    }, inplace=True)

embeddings = description_embeddings.merge(description_llm_embeddings, on="ID")

In [99]:
bills_llm_completion = bills_llm_completion.merge(embeddings, on="ID")

In [100]:
# bills_llm_completion_embed = bills_llm_completion.merge(embedded_descriptions, on="ID").merge(embedded_descriptions_llm, on="ID")
# bills_llm_completion_embed.sort_index(inplace=True)
print(len(bills_llm_completion))

39999


In [101]:
import random

def randomly_paired_similarity(df, N=10e3):
    true_sample = df[["DescriptionCleanEmbed"]].sample(n=int(N), replace=True).reset_index(drop=True)
    pred_sample = df[["DescriptionLLMCleanEmbed"]].sample(n=int(N), replace=True).reset_index(drop=True)
    random_pairs = pd.concat([true_sample, pred_sample], axis=1)

    random_pairs["RandomBaseline.CosineSimilarity"] = random_pairs.apply(lambda x: cosine_similarity(x["DescriptionCleanEmbed"], x["DescriptionLLMCleanEmbed"]), axis=1)
    random_pairs["RandomBaseline.EuclideanDistance"] = random_pairs.apply(lambda x: euclidean_distance(x["DescriptionCleanEmbed"], x["DescriptionLLMCleanEmbed"]), axis=1)
    random_pairs = random_pairs[["RandomBaseline.CosineSimilarity", "RandomBaseline.EuclideanDistance"]]
    return(random_pairs.mean())

random.seed(123)
random_baseline_group = bills_llm_completion.groupby(['Model', 'AddIntrDate'])[['Model', 'AddIntrDate', 'DescriptionCleanEmbed', 'DescriptionLLMCleanEmbed']].apply(randomly_paired_similarity)
print(random_baseline_group)

                                RandomBaseline.CosineSimilarity  \
Model              AddIntrDate                                    
gpt-3.5-turbo-0125 False                               0.244772   
                   True                                0.243189   
gpt-4o-2024-05-13  False                               0.242129   
                   True                                0.243175   

                                RandomBaseline.EuclideanDistance  
Model              AddIntrDate                                    
gpt-3.5-turbo-0125 False                                1.227018  
                   True                                 1.228348  
gpt-4o-2024-05-13  False                                1.229154  
                   True                                 1.228342  


In [102]:
random_baseline_group_path = os.path.join(data_dir, "random_baseline_group.csv")
random_baseline_group.to_csv(random_baseline_group_path)
print(f"Saved {os.path.basename(random_baseline_group_path)}, n = {len(random_baseline_group)}, at {os.path.dirname(random_baseline_group_path)}")


Saved random_baseline_group.csv, n = 4, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Data/Prediction


# Embedded Matches

In [103]:
bills_llm_completion["EuclideanDistance"] = bills_llm_completion.apply(lambda x: euclidean_distance(x["DescriptionCleanEmbed"], x["DescriptionLLMCleanEmbed"]), axis=1)
bills_llm_completion["CosineSimilarity"] = bills_llm_completion.apply(lambda x: cosine_similarity(x["DescriptionCleanEmbed"], x["DescriptionLLMCleanEmbed"]), axis=1)


In [104]:
bills_llm_completion_similarity = bills_llm_completion[[
    'ID', 'BillID', 'PromptingStrategyID', 'PromptingStrategyName',
    'ResponseFormat', 'TrimText', 'AddIntrDate', 'Model', 'Temperature',
    'MaxTokens', 'Year', 'Major', 'MajorText', 'Party', 'Chamber', 'DW1', 'PassH', 'PassS', 'Postal', 'IntrDate',
    'DescriptionTrim', 'Description', 'DescriptionLLM',
    'CosineSimilarity', 'EuclideanDistance', 'TextSimilarity']]
bills_llm_completion_similarity_path = os.path.join(data_dir, "bills_llm_completion_similarity.csv")
bills_llm_completion_similarity.to_csv(bills_llm_completion_similarity_path, index=False)
print(f"Saved {os.path.basename(bills_llm_completion_similarity_path)}, n = {len(bills_llm_completion_similarity)}, at {os.path.dirname(bills_llm_completion_similarity_path)}")

Saved bills_llm_completion_similarity.csv, n = 39999, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Data/Prediction
